In [1]:
import pandas as pd
import numpy as np
import torch
import re
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

/home/nexpg/anaconda3/envs/my314/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# 1. Функция очистки текста
# ----------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:()\[\]{}\'"-+=/&*#@$%~`_—-]'
    cleaned = re.sub(pattern, '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

In [3]:
# ----------------------------
# 2. Загрузка словаря микрокатегорий
# ----------------------------
with open('key_phrases.json', 'r', encoding='utf-8') as f:
    key_phrases = json.load(f)
if isinstance(key_phrases, list):
    mc_id_to_title = {item['mcId']: item['mcTitle'] for item in key_phrases}
else:
    mc_id_to_title = {int(k): v for k, v in key_phrases.items()}
all_mc_ids = sorted(mc_id_to_title.keys())
id2idx = {mc_id: i for i, mc_id in enumerate(all_mc_ids)}
idx2id = {i: mc_id for mc_id, i in id2idx.items()}


In [4]:
# ----------------------------
# 3. Загрузка моделей и токенизатора
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained("deepvk/USER2-base")
detected_model = AutoModelForSequenceClassification.from_pretrained(
    "./final_detected_model",
    device_map="auto"
)
detected_model.eval()
split_model = AutoModelForSequenceClassification.from_pretrained(
    "./final_split_model",
    device_map="auto"
)
split_model.eval()

DETECTED_THRESHOLD = 0.45
SPLIT_THRESHOLD = 0.25


Loading weights: 100%|██████████| 138/138 [00:00<00:00, 292.66it/s]


In [5]:
# ----------------------------
# 4. Функция предсказания меток
# ----------------------------
def predict_labels(model, texts, threshold, batch_size=16):
    predictions = []
    device = next(model.parameters()).device
    for i in tqdm(range(0, len(texts), batch_size), desc="Инференс", unit="batch"):
        batch_texts = texts[i:i+batch_size]
        encodings = tokenizer(batch_texts, truncation=True, max_length=1024,
                              padding=True, return_tensors="pt")
        encodings = {k: v.to(device) for k, v in encodings.items()}
        with torch.no_grad():
            logits = model(**encodings).logits
            probs = torch.sigmoid(logits).cpu().numpy()
        for prob in probs:
            pred_ids = [idx2id[i] for i, p in enumerate(prob) if p > threshold]
            predictions.append(pred_ids)
    return predictions


In [6]:
# ----------------------------
# 5. Обработка тестового файла
# ----------------------------
df_test = pd.read_csv('rnc_test.csv')

# Извлекаем description из request
descriptions = []
requests = []
for req_str in df_test['request']:
    req = json.loads(req_str)
    requests.append(req)
    descriptions.append(req['description'])

# Очистка текста
cleaned_descriptions = [clean_text(desc) for desc in descriptions]

# Предсказания detected
test_texts_detected = [f"classification: {desc}" for desc in cleaned_descriptions]
print("Предсказания detected модели...")
pred_detected = predict_labels(detected_model, test_texts_detected, DETECTED_THRESHOLD, batch_size=16)

# Формируем тексты для split модели
test_texts_split = []
for idx, desc in enumerate(cleaned_descriptions):
    pred_mc_list = pred_detected[idx]
    detected_str = ", ".join(map(str, pred_mc_list)) if pred_mc_list else "none"
    text_split = f"categories: {detected_str} [SEP] {desc}"
    test_texts_split.append(text_split)

print("Предсказания split модели...")
pred_split = predict_labels(split_model, test_texts_split, SPLIT_THRESHOLD, batch_size=16)


Предсказания detected модели...


Инференс: 100%|██████████| 10/10 [00:10<00:00,  1.09s/batch]


Предсказания split модели...


Инференс: 100%|██████████| 10/10 [00:10<00:00,  1.05s/batch]


In [7]:
# ----------------------------
# 6. Формирование ответов
# ----------------------------
responses = []
for idx, req in enumerate(requests):
    detected_ids = pred_detected[idx]
    split_ids = pred_split[idx]
    should_split = len(split_ids) > 0
    drafts = []
    for mc_id in split_ids:
        drafts.append({
            "mcId": mc_id,
            "mcTitle": mc_id_to_title.get(mc_id, f"Категория {mc_id}"),
            "text": ""   # пустой текст
        })
    response = {
        "detectedMcIds": detected_ids,
        "shouldSplit": should_split,
        "drafts": drafts
    }
    responses.append(json.dumps(response, ensure_ascii=False))

df_test['response'] = responses

# Сохраняем результат
df_test.to_csv('rnc_test_with_predictions.csv', index=False)
print("Готово! Результат сохранён в rnc_test_with_predictions.csv")

Готово! Результат сохранён в rnc_test_with_predictions.csv
